In [1]:
from tqdm.auto import tqdm
import os
import json
import csv
import hashlib
import re
import random
from pathlib import Path
from typing import Any
from decimal import Decimal
from datetime import datetime, timezone
from openai import OpenAI


# ============================================================
# CONFIGURAÇÃO
# ============================================================

INPUT_FILE = Path("geracao.jsonl")
OUTPUT_DIR = Path("topic_generation")

MODEL = "gpt-6.1-sol"
BATCH_SIZE = 100
REASONING_EFFORT = "low"

PROMPT_VERSION = "topic-generation-v3"

# Número máximo de tentativas por lote quando a resposta é inválida.
MAX_ATTEMPTS = 3

# Os caminhos específicos são definidos separadamente para cada experimento.
STATE_FILE = OUTPUT_DIR / "state.json"
BATCH_DIR = OUTPUT_DIR / "batches"
FINAL_FILE = OUTPUT_DIR / "topics_final.json"
COST_REPORT = OUTPUT_DIR / "costs.csv"
LOG_DIR = OUTPUT_DIR / "logs"

# Semente fixa para reproduzir exatamente a segunda ordem.
SHUFFLE_SEED = 20261026

# Preços padrão (USD por 1 milhão de tokens), GPT-6.1 Sol.
# Estes valores correspondem ao processamento Standard e a prompts
# com até 272 mil tokens de entrada.
PRICES = {
    "input": Decimal("2.00"),
    "cached_input": Decimal("0.10"),
    "cache_write_30m": Decimal("2.50"),
    "cache_write_12h": Decimal("5.00"),
    "output": Decimal("10.00"),
}

if not os.getenv("OPENAI_API_KEY"):
    raise EnvironmentError(
        "A variável de ambiente OPENAI_API_KEY não foi encontrada."
    )

client = OpenAI(
    max_retries=5,
    timeout=180.0
)

In [2]:
# ============================================================
# PROMPT E SAÍDA ESTRUTURADA
# ============================================================

SYSTEM_PROMPT = """
Você é um pesquisador especializado em análise temática. Sua tarefa é identificar novos tópicos em títulos de vídeos do YouTube
sobre medicamentos para emagrecimento. Um tópico representa um assunto temático recorrente, e não apenas uma palavra ou medicamento mencionado.

Você receberá:
1. Os tópicos identificados nos lotes anteriores, um por linha, no formato - rótulo: descrição.
2. Um novo lote de títulos, um por linha, no formato video_id<TAB>título (separados por tabulação).

Regras:
- Considere os tópicos existentes antes de propor um novo.
- Crie tópicos somente quando houver um tema relevante que ainda não
  seja contemplado pelos tópicos existentes.
- Não crie tópicos apenas por diferenças de palavras, medicamentos ou
  estilo dos títulos.
- Evite tópicos redundantes ou excessivamente específicos.
- Não altere, renomeie, combine ou exclua tópicos existentes.
- Considere o assunto do título, não a veracidade de suas
  afirmações.
- Não force a criação de tópicos: se nenhum tema novo for necessário,
  retorne uma lista vazia.
- Os nomes e descrições devem estar em português.

Para cada tópico novo, forneça:
- label: nome curto e conciso.
- description: descrição em uma frase.
- evidence_video_ids: exemplos de IDs de um ou mais títulos do lote que
  justificam a criação do tópico.

Retorne somente os tópicos novos.
"""

OUTPUT_SCHEMA = {
    "type": "object",
    "properties": {
        "new_topics": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "label": {"type": "string"},
                    "description": {"type": "string"},
                    "evidence_video_ids": {
                        "type": "array",
                        "items": {"type": "string"}
                    }
                },
                "required": [
                    "label",
                    "description",
                    "evidence_video_ids"
                ],
                "additionalProperties": False
            }
        }
    },
    "required": ["new_topics"],
    "additionalProperties": False
}


# ============================================================
# FUNÇÕES DE ARQUIVO E HASH
# ============================================================

def sha256_text(value: str) -> str:
    return hashlib.sha256(value.encode("utf-8")).hexdigest()


def canonical_json(data: Any) -> str:
    return json.dumps(
        data,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":")
    )


def save_json_atomic(path: Path, data: Any) -> None:
    """Grava JSON por substituição atômica para reduzir risco de corrupção."""
    path.parent.mkdir(parents=True, exist_ok=True)
    temp_path = path.with_name(path.name + ".tmp")

    with open(temp_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
        f.flush()
        os.fsync(f.fileno())

    os.replace(temp_path, path)


def load_json(path: Path) -> Any:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def normalize_label(label: str) -> str:
    return re.sub(r"\s+", " ", label.strip().casefold())


# ============================================================
# LEITURA DOS DADOS
# ============================================================

def load_titles(path: Path) -> list[dict]:
    if not path.exists():
        raise FileNotFoundError(f"Arquivo de entrada não encontrado: {path}")

    records = []

    with open(path, "r", encoding="utf-8-sig") as f:
        for line_number, line in enumerate(f, start=1):
            if not line.strip():
                continue

            try:
                item = json.loads(line)
            except json.JSONDecodeError as exc:
                raise ValueError(
                    f"JSON inválido na linha {line_number}: {exc}"
                ) from exc

            if not isinstance(item, dict):
                raise ValueError(
                    f"A linha {line_number} não contém um objeto JSON."
                )

            if "video_id" not in item or "title" not in item:
                raise ValueError(
                    f"Linha {line_number} sem os campos title e/ou video_id."
                )

            video_id = str(item["video_id"]).strip()
            title = " ".join(str(item["title"]).split())

            if not video_id or not title:
                raise ValueError(
                    f"video_id ou title vazio na linha {line_number}."
                )

            if re.search(r"\s", video_id):
                raise ValueError(
                    f"video_id com espaço ou tabulação na linha {line_number}."
                )

            records.append({
                "video_id": video_id,
                "title": title
            })

    if not records:
        raise ValueError("Nenhum título válido foi encontrado.")

    ids = [record["video_id"] for record in records]
    if len(ids) != len(set(ids)):
        duplicates = sorted({
            video_id for video_id in ids if ids.count(video_id) > 1
        })
        raise ValueError(
            f"Existem video_id duplicados. Exemplos: {duplicates[:10]}"
        )

    return records


def create_batches(records: list[dict]) -> list[list[dict]]:
    return [
        records[i:i + BATCH_SIZE]
        for i in range(0, len(records), BATCH_SIZE)
    ]


# ============================================================
# CHAMADA À API
# ============================================================

def create_user_prompt(
    existing_topics: list[dict],
    batch: list[dict]
) -> str:
    if existing_topics:
        topics_text = "\n".join(
            f"- {t['label']}: {t['description']}"
            for t in existing_topics
        )
    else:
        topics_text = "(nenhum tópico ainda)"

    titles_text = "\n".join(
        f"{r['video_id']}\t{r['title']}"
        for r in batch
    )

    return f"""TÓPICOS IDENTIFICADOS ATÉ AGORA ({len(existing_topics)}):
{topics_text}

NOVO LOTE DE TÍTULOS (video_id<TAB>título):
{titles_text}

Identifique somente tópicos novos que sejam necessários.
Use exclusivamente video_id do lote atual como evidência."""


def request_model(user_prompt: str):
    return client.responses.create(
        model=MODEL,
        instructions=SYSTEM_PROMPT,
        input=user_prompt,
        reasoning={"effort": REASONING_EFFORT},
        text={
            "format": {
                "type": "json_schema",
                "name": "topic_generation",
                "strict": True,
                "schema": OUTPUT_SCHEMA
            }
        }
    )


def parse_response(response) -> dict:
    if response.status != "completed":
        raise RuntimeError(
            f"Resposta da API não concluída. "
            f"Status: {response.status}; "
            f"detalhes: {response.incomplete_details}"
        )

    if not response.output_text:
        raise RuntimeError(
            "A API concluiu a resposta, mas não retornou texto de saída."
        )

    try:
        return json.loads(response.output_text)
    except json.JSONDecodeError as exc:
        raise ValueError(
            "A saída da API não pôde ser interpretada como JSON."
        ) from exc


# ============================================================
# VALIDAÇÃO DOS RESULTADOS
# ============================================================

def validate_result(
    result: dict,
    batch: list[dict],
    existing_topics: list[dict]
) -> None:
    if not isinstance(result, dict) or set(result) != {"new_topics"}:
        raise ValueError("Estrutura de resposta inesperada.")

    if not isinstance(result["new_topics"], list):
        raise ValueError("'new_topics' precisa ser uma lista.")

    batch_ids = {record["video_id"] for record in batch}
    existing_labels = {
        normalize_label(topic["label"])
        for topic in existing_topics
    }
    new_labels = set()

    for index, topic in enumerate(result["new_topics"], start=1):
        if not isinstance(topic, dict):
            raise ValueError(f"Tópico {index} não é um objeto.")

        label = topic.get("label")
        description = topic.get("description")
        evidence = topic.get("evidence_video_ids")

        if not isinstance(label, str) or not label.strip():
            raise ValueError(f"Tópico {index} possui label inválido.")

        if not isinstance(description, str) or not description.strip():
            raise ValueError(f"Tópico '{label}' possui descrição inválida.")

        if not isinstance(evidence, list) or not evidence:
            raise ValueError(
                f"O tópico '{label}' não possui evidências válidas."
            )

        if not all(isinstance(video_id, str) for video_id in evidence):
            raise ValueError(
                f"O tópico '{label}' contém IDs que não são strings."
            )

        normalized = normalize_label(label)

        if normalized in existing_labels:
            raise ValueError(
                f"O modelo tentou recriar tópico existente: {label}"
            )

        if normalized in new_labels:
            raise ValueError(
                f"Tópicos duplicados na mesma resposta: {label}"
            )

        new_labels.add(normalized)

        invalid_ids = set(evidence) - batch_ids
        if invalid_ids:
            raise ValueError(
                f"O tópico '{label}' contém IDs fora do lote atual: "
                f"{sorted(invalid_ids)}"
            )

        if len(evidence) != len(set(evidence)):
            raise ValueError(
                f"O tópico '{label}' contém evidências duplicadas."
            )


# ============================================================
# CÁLCULO DO CUSTO
# ============================================================

def calculate_cost(usage: dict) -> dict:
    """
    Calcula o custo estimado em USD a partir do usage retornado pela API.

    O cálculo considera:
    - entrada normal;
    - entrada lida do cache;
    - gravações de cache de 30 minutos e 12 horas;
    - tokens de saída.

    Os tokens de raciocínio já estão incluídos em output_tokens.
    """

    input_tokens = int(usage["input_tokens"])
    output_tokens = int(usage["output_tokens"])

    input_details = usage.get("input_tokens_details") or {}

    cached_tokens = int(input_details.get("cached_tokens") or 0)
    cache_write_tokens = int(input_details.get("cache_write_tokens") or 0)

    write_details = input_details.get("cache_write_tokens_details") or {}

    if write_details:
        write_30m = int(write_details.get("ttl_30m_tokens") or 0)
        write_12h = int(write_details.get("ttl_12h_tokens") or 0)

        if write_30m + write_12h != cache_write_tokens:
            raise ValueError(
                "Detalhamento de cache inconsistente: "
                "ttl_30m_tokens + ttl_12h_tokens != cache_write_tokens."
            )
    else:
        # Nesta configuração não definimos prompt_cache_options.ttl,
        # portanto vale a gravação padrão de 30 minutos.
        write_30m = cache_write_tokens
        write_12h = 0

    ordinary_input = input_tokens - cached_tokens - cache_write_tokens

    if min(input_tokens, output_tokens, cached_tokens,
           cache_write_tokens, write_30m, write_12h) < 0:
        raise ValueError("A API retornou contagem negativa de tokens.")

    if ordinary_input < 0:
        raise ValueError("Detalhamento de tokens de entrada inconsistente.")

    # Preços padrão para prompts de até 272 mil tokens.
    input_price = PRICES["input"]
    cached_price = PRICES["cached_input"]
    write_30m_price = PRICES["cache_write_30m"]
    write_12h_price = PRICES["cache_write_12h"]
    output_price = PRICES["output"]

    # Sobretaxa de contexto longo, aplicada à requisição inteira.
    if input_tokens > 272_000:
        input_price *= 2
        cached_price *= 2
        write_30m_price *= 2
        write_12h_price *= 2
        output_price *= Decimal("1.5")

    million = Decimal("1000000")

    input_cost = (
        Decimal(ordinary_input) * input_price
        + Decimal(cached_tokens) * cached_price
        + Decimal(write_30m) * write_30m_price
        + Decimal(write_12h) * write_12h_price
    ) / million

    output_cost = (
        Decimal(output_tokens) * output_price
    ) / million

    output_details = usage.get("output_tokens_details") or {}

    return {
        "input_tokens": input_tokens,
        "ordinary_input_tokens": ordinary_input,
        "cached_tokens": cached_tokens,
        "cache_write_tokens": cache_write_tokens,
        "cache_write_30m_tokens": write_30m,
        "cache_write_12h_tokens": write_12h,
        "output_tokens": output_tokens,
        "reasoning_tokens": int(output_details.get("reasoning_tokens") or 0),
        "input_cost_usd": str(input_cost),
        "output_cost_usd": str(output_cost),
        "total_cost_usd": str(input_cost + output_cost),
        "pricing_basis": "standard"
    }


# ============================================================
# LOGS EM TEXTO
# ============================================================

def save_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)
        f.flush()
        os.fsync(f.fileno())


def next_log_path(batch_index: int) -> Path:
    """Próximo nome livre; nunca sobrescreve logs de execuções anteriores."""
    LOG_DIR.mkdir(parents=True, exist_ok=True)
    existing = list(LOG_DIR.glob(f"batch_{batch_index:03d}_call_*.txt"))
    return LOG_DIR / f"batch_{batch_index:03d}_call_{len(existing) + 1:02d}.txt"


def write_call_log(
    batch_index: int,
    attempt: int,
    user_prompt: str,
    response,
    status: str
) -> Path:
    """Grava prompt enviado, saída bruta do modelo, uso e resultado da validação."""
    path = next_log_path(batch_index)
    sep = "=" * 70

    usage = getattr(response, "usage", None)
    usage_text = (
        json.dumps(usage.model_dump(), ensure_ascii=False, indent=2, default=str)
        if usage is not None else "(sem dados de uso)"
    )

    if response is None:
        output_text = "(nenhuma resposta recebida)"
    else:
        output_text = getattr(response, "output_text", None) or "(saída vazia)"

    text = f"""{sep}
LOG DA CHAMADA
{sep}
Experimento: {OUTPUT_DIR.name}
Lote: {batch_index}
Tentativa: {attempt}/{MAX_ATTEMPTS}
Data/hora (UTC): {datetime.now(timezone.utc).isoformat()}
Modelo: {MODEL}
Reasoning effort: {REASONING_EFFORT}
Versão do prompt: {PROMPT_VERSION}
Response ID: {getattr(response, "id", None)}
Status da API: {getattr(response, "status", None)}
Validação: {status}

{sep}
INSTRUÇÕES (SYSTEM PROMPT)
{sep}
{SYSTEM_PROMPT.strip()}

{sep}
PROMPT DO USUÁRIO
{sep}
{user_prompt}

{sep}
SAÍDA DO MODELO
{sep}
{output_text}

{sep}
USO (usage)
{sep}
{usage_text}
"""
    save_text(path, text)
    return path


def write_config_log(
    experiment_name: str,
    order_description: str,
    shuffle_seed,
    input_hash: str
) -> None:
    sep = "=" * 70
    text = f"""{sep}
CONFIGURAÇÃO DO EXPERIMENTO
{sep}
Experimento: {experiment_name}
Ordem: {order_description}
Semente do embaralhamento: {shuffle_seed}
Hash da entrada: {input_hash}
Modelo: {MODEL}
Reasoning effort: {REASONING_EFFORT}
Tamanho do lote: {BATCH_SIZE}
Máximo de tentativas: {MAX_ATTEMPTS}
Versão do prompt: {PROMPT_VERSION}

{sep}
INSTRUÇÕES (SYSTEM PROMPT)
{sep}
{SYSTEM_PROMPT.strip()}

{sep}
SCHEMA DE SAÍDA
{sep}
{json.dumps(OUTPUT_SCHEMA, ensure_ascii=False, indent=2)}
"""
    save_text(LOG_DIR / "00_config.txt", text)


def generate_with_retries(
    existing_topics: list[dict],
    batch: list[dict],
    batch_index: int
):
    """
    Chama o modelo e valida a resposta, com até MAX_ATTEMPTS tentativas.

    Retorna (result, response, failed_attempts). As tentativas que falharam
    guardam o usage, o custo e o caminho do log, para que o relatório não
    subestime o gasto. Toda chamada gera um log .txt, inclusive as que falham
    por erro de rede/SDK (esses erros são registrados e depois relançados).
    """
    failed_attempts = []
    last_error = None
    user_prompt = create_user_prompt(existing_topics, batch)

    for attempt in range(1, MAX_ATTEMPTS + 1):
        response = None
        try:
            response = request_model(user_prompt)
            result = parse_response(response)
            validate_result(result, batch, existing_topics)

            if response.usage is None:
                raise RuntimeError(
                    "A API não retornou dados de uso; lote não será concluído."
                )

            write_call_log(batch_index, attempt, user_prompt, response, "OK")
            return result, response, failed_attempts

        except (ValueError, RuntimeError) as exc:
            last_error = exc
            log_path = write_call_log(
                batch_index, attempt, user_prompt, response, f"FALHOU: {exc}"
            )
            record = {
                "attempt": attempt,
                "error": str(exc),
                "response_id": getattr(response, "id", None),
                "log_file": str(log_path)
            }

            usage = getattr(response, "usage", None)
            if usage is not None:
                usage_data = usage.model_dump()
                record["usage"] = usage_data
                record["cost"] = calculate_cost(usage_data)

            failed_attempts.append(record)
            print(
                f"Tentativa {attempt}/{MAX_ATTEMPTS} do lote "
                f"{batch_index} falhou: {exc}"
            )

        except Exception as exc:
            write_call_log(
                batch_index, attempt, user_prompt, response,
                f"ERRO DA API/SDK: {type(exc).__name__}: {exc}"
            )
            raise

    raise RuntimeError(
        f"Lote {batch_index} falhou após {MAX_ATTEMPTS} tentativas. "
        f"Último erro: {last_error}"
    ) from last_error


def failed_attempts_cost(saved: dict) -> Decimal:
    """Soma o custo das tentativas que falharam num checkpoint."""
    return sum(
        (
            Decimal(attempt["cost"]["total_cost_usd"])
            for attempt in saved.get("failed_attempts", [])
            if attempt.get("cost")
        ),
        Decimal("0")
    )


# ============================================================
# ESTADO E CHECKPOINTS
# ============================================================

def initialize_state(input_hash: str, config_hash: str) -> dict:
    return {
        "input_hash": input_hash,
        "config_hash": config_hash,
        "model": MODEL,
        "batch_size": BATCH_SIZE,
        "reasoning_effort": REASONING_EFFORT,
        "prompt_version": PROMPT_VERSION,
        "topics": [],
        "processed_batches": [],
        "total_batches": 0,
        "created_at": datetime.now(timezone.utc).isoformat(),
        "updated_at": datetime.now(timezone.utc).isoformat()
    }


def load_or_initialize_state(input_hash: str, config_hash: str) -> dict:
    if not STATE_FILE.exists():
        return initialize_state(input_hash, config_hash)

    state = load_json(STATE_FILE)

    expected = {
        "input_hash": input_hash,
        "config_hash": config_hash,
        "model": MODEL,
        "batch_size": BATCH_SIZE,
        "reasoning_effort": REASONING_EFFORT,
        "prompt_version": PROMPT_VERSION
    }

    for key, value in expected.items():
        if state.get(key) != value:
            raise RuntimeError(
                f"O estado salvo é incompatível com a configuração atual "
                f"(campo: {key}). Use outro OUTPUT_DIR ou arquive a execução."
            )

    if not isinstance(state.get("topics"), list):
        raise RuntimeError("state.json contém uma lista de tópicos inválida.")

    if not isinstance(state.get("processed_batches"), list):
        raise RuntimeError("state.json contém processed_batches inválido.")

    return state


def apply_batch_result(
    state: dict,
    result: dict,
    batch_index: int
) -> None:
    if batch_index in state["processed_batches"]:
        raise RuntimeError(f"Lote {batch_index} já consta como processado.")

    for topic in result["new_topics"]:
        topic_id = f"T{len(state['topics']) + 1:03d}"

        state["topics"].append({
            "topic_id": topic_id,
            "label": topic["label"].strip(),
            "description": topic["description"].strip(),
            "evidence_video_ids": topic["evidence_video_ids"],
            "created_in_batch": batch_index
        })

    state["processed_batches"].append(batch_index)
    state["processed_batches"].sort()
    state["updated_at"] = datetime.now(timezone.utc).isoformat()


# ============================================================
# RELATÓRIO DE CUSTOS
# ============================================================

def export_cost_report() -> None:
    rows = []

    for path in sorted(BATCH_DIR.glob("batch_*.json")):
        saved = load_json(path)
        usage = saved.get("usage")

        if usage is None:
            continue

        cost = saved.get("cost")
        if cost is None:
            cost = calculate_cost(usage)

        rows.append({
            "batch": saved["batch_index"],
            "response_id": saved.get("response_id", ""),
            "titles_in_batch": saved.get("titles_in_batch", ""),
            "new_topics": len(saved.get("result", {}).get("new_topics", [])),
            "input_tokens": cost["input_tokens"],
            "ordinary_input_tokens": cost["ordinary_input_tokens"],
            "cached_tokens": cost["cached_tokens"],
            "cache_write_tokens": cost["cache_write_tokens"],
            "cache_write_30m_tokens": cost["cache_write_30m_tokens"],
            "cache_write_12h_tokens": cost["cache_write_12h_tokens"],
            "output_tokens": cost["output_tokens"],
            "reasoning_tokens": cost["reasoning_tokens"],
            "input_cost_usd": cost["input_cost_usd"],
            "output_cost_usd": cost["output_cost_usd"],
            "total_cost_usd": cost["total_cost_usd"],
            "pricing_basis": cost["pricing_basis"],
            "failed_attempts": len(saved.get("failed_attempts", [])),
            "failed_attempts_cost_usd": str(failed_attempts_cost(saved))
        })

    fieldnames = [
        "batch", "response_id", "titles_in_batch", "new_topics",
        "input_tokens", "ordinary_input_tokens", "cached_tokens",
        "cache_write_tokens", "cache_write_30m_tokens",
        "cache_write_12h_tokens", "output_tokens", "reasoning_tokens",
        "input_cost_usd", "output_cost_usd", "total_cost_usd",
        "pricing_basis", "failed_attempts", "failed_attempts_cost_usd"
    ]

    COST_REPORT.parent.mkdir(parents=True, exist_ok=True)

    with open(COST_REPORT, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

    total_cost = sum(
        (
            Decimal(row["total_cost_usd"])
            + Decimal(row["failed_attempts_cost_usd"])
            for row in rows
        ),
        Decimal("0")
    )

    print(f"Relatório de custos atualizado: {COST_REPORT}")
    print(
        f"Custo acumulado estimado (inclui tentativas falhas): "
        f"US$ {total_cost:.8f}"
    )


# ============================================================
# EXECUÇÃO PRINCIPAL
# ============================================================

def run_experiment(
    experiment_name: str,
    records: list[dict],
    order_description: str,
    shuffle_seed: int | None = None
) -> None:
    """
    Executa uma taxonomia independente, com estado e checkpoints próprios.
    """
    global OUTPUT_DIR, STATE_FILE, BATCH_DIR, FINAL_FILE, COST_REPORT, LOG_DIR

    experiment_dir = Path("topic_generation") / experiment_name
    OUTPUT_DIR = experiment_dir
    STATE_FILE = OUTPUT_DIR / "state.json"
    BATCH_DIR = OUTPUT_DIR / "batches"
    FINAL_FILE = OUTPUT_DIR / "topics_final.json"
    COST_REPORT = OUTPUT_DIR / "costs.csv"
    LOG_DIR = OUTPUT_DIR / "logs"

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    BATCH_DIR.mkdir(parents=True, exist_ok=True)
    LOG_DIR.mkdir(parents=True, exist_ok=True)

    batches = create_batches(records)
    input_hash = sha256_text(canonical_json(records))

    experiment_metadata = {
        "experiment": experiment_name,
        "order_description": order_description,
        "shuffle_seed": shuffle_seed,
        "input_hash": input_hash,
        "total_titles": len(records)
    }

    # Manifesto permite auditar a sequência efetivamente processada.
    save_json_atomic(OUTPUT_DIR / "order_manifest.json", {
        **experiment_metadata,
        "video_ids_in_order": [record["video_id"] for record in records]
    })

    config_for_hash = {
        "model": MODEL,
        "batch_size": BATCH_SIZE,
        "reasoning_effort": REASONING_EFFORT,
        "prompt_version": PROMPT_VERSION,
        "system_prompt": SYSTEM_PROMPT,
        "output_schema": OUTPUT_SCHEMA,
        "prices": {key: str(value) for key, value in PRICES.items()},
        "experiment": experiment_metadata
    }
    config_hash = sha256_text(canonical_json(config_for_hash))

    state = load_or_initialize_state(input_hash, config_hash)
    state["total_batches"] = len(batches)
    state["experiment"] = experiment_metadata

    processed = set(state["processed_batches"])
    expected_batch_numbers = set(range(1, len(batches) + 1))

    if not processed.issubset(expected_batch_numbers):
        raise RuntimeError(
            "state.json contém números de lotes fora do intervalo esperado."
        )

    if processed:
        expected_processed = set(range(1, max(processed) + 1))
        if processed != expected_processed:
            raise RuntimeError(
                "state.json contém lotes concluídos fora de sequência. "
                "Verifique os checkpoints antes de continuar."
            )

    for batch_index in processed:
        checkpoint = BATCH_DIR / f"batch_{batch_index:03d}.json"
        if not checkpoint.exists():
            raise RuntimeError(
                f"Lote {batch_index} consta como concluído no estado, "
                "mas seu checkpoint não existe."
            )

    save_json_atomic(STATE_FILE, state)
    write_config_log(experiment_name, order_description, shuffle_seed, input_hash)

    print("\n" + "=" * 70)
    print(f"EXPERIMENTO: {experiment_name}")
    print(f"Ordem: {order_description}")
    if shuffle_seed is not None:
        print(f"Semente do embaralhamento: {shuffle_seed}")
    print("=" * 70)
    print(f"Modelo: {MODEL}")
    print(f"Títulos: {len(records)}")
    print(f"Tamanho dos lotes: {BATCH_SIZE}")
    print(f"Total de lotes: {len(batches)}")
    print(f"Tópicos existentes nesta taxonomia: {len(state['topics'])}")
    print(f"Lotes já concluídos: {len(processed)}")
    print(f"Diretório: {OUTPUT_DIR}")
    print("=" * 70)

    progress = tqdm(
    total=len(batches),
    initial=len(processed),
    desc=experiment_name,
    unit="lote"
    )
    for batch_index, batch in enumerate(batches, start=1):
        batch_path = BATCH_DIR / f"batch_{batch_index:03d}.json"

        if batch_index in state["processed_batches"]:
            print(
                f"[{batch_index}/{len(batches)}] "
                "Lote já processado; ignorando."
            )
            continue

        request_fingerprint = sha256_text(canonical_json({
            "batch": batch,
            "existing_topics": state["topics"],
            "model": MODEL,
            "reasoning_effort": REASONING_EFFORT,
            "prompt_version": PROMPT_VERSION,
            "system_prompt": SYSTEM_PROMPT,
            "output_schema": OUTPUT_SCHEMA,
            "experiment": experiment_metadata
        }))

        print("\n" + "-" * 60)
        print(f"LOTE {batch_index}/{len(batches)}")
        print(f"Títulos: {len(batch)}")
        print(f"Tópicos antes da chamada: {len(state['topics'])}")
        print("-" * 60)

        # Recupera resposta salva se houve interrupção após o checkpoint.
        if batch_path.exists():
            saved = load_json(batch_path)

            if saved.get("request_fingerprint") != request_fingerprint:
                raise RuntimeError(
                    f"Checkpoint incompatível no lote {batch_index}. "
                    "Não será sobrescrito automaticamente."
                )

            result = saved["result"]
            usage_data = saved.get("usage")

            if usage_data is None:
                raise RuntimeError(
                    f"Checkpoint do lote {batch_index} não contém usage."
                )

            cost_data = saved.get("cost")
            if cost_data is None:
                cost_data = calculate_cost(usage_data)
                saved["cost"] = cost_data
                save_json_atomic(batch_path, saved)

            print("Resposta recuperada do checkpoint.")

        else:
            print("Enviando chamada à API...")
            result, response, failed_attempts = generate_with_retries(
                state["topics"], batch, batch_index
            )

            usage_data = response.usage.model_dump()
            cost_data = calculate_cost(usage_data)

            save_json_atomic(batch_path, {
                "batch_index": batch_index,
                "request_fingerprint": request_fingerprint,
                "response_id": response.id,
                "model": response.model,
                "created_at": datetime.now(timezone.utc).isoformat(),
                "titles_in_batch": len(batch),
                "usage": usage_data,
                "cost": cost_data,
                "failed_attempts": failed_attempts,
                "result": result
            })

        validate_result(result, batch, state["topics"])

        previous_count = len(state["topics"])
        apply_batch_result(state, result, batch_index)
        new_count = len(state["topics"]) - previous_count

        save_json_atomic(STATE_FILE, state)
        export_cost_report()
        progress.update(1)

        progress.set_postfix({
        "tópicos": len(state["topics"]),
        "custo": f"US$ {Decimal(cost_data['total_cost_usd']):.4f}"
        })

        print(f"Novos tópicos: {new_count}")
        print(f"Total acumulado: {len(state['topics'])}")
        print(f"Tokens de entrada: {cost_data['input_tokens']:,}")
        print(f"Tokens em cache: {cost_data['cached_tokens']:,}")
        print(f"Tokens de saída: {cost_data['output_tokens']:,}")
        print(
            f"Custo estimado do lote: "
            f"US$ {Decimal(cost_data['total_cost_usd']):.8f}"
        )
    progress.close()
    save_json_atomic(FINAL_FILE, {
        "model": MODEL,
        "reasoning_effort": REASONING_EFFORT,
        "prompt_version": PROMPT_VERSION,
        "experiment": experiment_metadata,
        "input_file": str(INPUT_FILE),
        "input_hash": input_hash,
        "config_hash": config_hash,
        "total_titles": len(records),
        "batch_size": BATCH_SIZE,
        "total_batches": len(batches),
        "total_topics": len(state["topics"]),
        "topics": state["topics"]
    })

    export_cost_report()

    total_cost = Decimal("0")
    for batch_index in range(1, len(batches) + 1):
        checkpoint_path = BATCH_DIR / f"batch_{batch_index:03d}.json"
        if not checkpoint_path.exists():
            raise RuntimeError(
                f"Checkpoint ausente para o lote {batch_index}."
            )

        checkpoint = load_json(checkpoint_path)
        total_cost += Decimal(checkpoint["cost"]["total_cost_usd"])
        total_cost += failed_attempts_cost(checkpoint)

    print("\n" + "=" * 70)
    print(f"EXPERIMENTO CONCLUÍDO: {experiment_name}")
    print("=" * 70)
    print(f"Títulos processados: {len(records)}")
    print(f"Lotes: {len(batches)}")
    print(f"Tópicos identificados: {len(state['topics'])}")
    print(f"Custo total estimado (inclui tentativas falhas): US$ {total_cost:.8f}")
    print(
        f"Custo médio por título: "
        f"US$ {total_cost / Decimal(len(records)):.8f}"
    )
    print(f"Taxonomia final: {FINAL_FILE}")
    print(f"Relatório de custos: {COST_REPORT}")
    print(f"Estado/checkpoints: {OUTPUT_DIR}")
    print(f"Logs de prompts e saídas: {LOG_DIR}")
    print("=" * 70)


def main():
    # Primeiro experimento: ordem original do arquivo.
    original_records = load_titles(INPUT_FILE)

    run_experiment(
        experiment_name="experiment_1_original_order",
        records=original_records,
        order_description="ordem original do arquivo JSONL"
    )

    # Só começa depois de o primeiro experimento ter sido concluído.
    # Os mesmos registros são embaralhados; nada é removido ou alterado.
    shuffled_records = list(original_records)
    random.Random(SHUFFLE_SEED).shuffle(shuffled_records)

    if [r["video_id"] for r in shuffled_records] == [
        r["video_id"] for r in original_records
    ] and len(original_records) > 1:
        raise RuntimeError(
            "O embaralhamento produziu a mesma ordem original. "
            "Altere SHUFFLE_SEED."
        )

    run_experiment(
        experiment_name="experiment_2_shuffled_order",
        records=shuffled_records,
        order_description="ordem embaralhada deterministicamente",
        shuffle_seed=SHUFFLE_SEED
    )

    print("\nOS DOIS EXPERIMENTOS FORAM CONCLUÍDOS.")
    print("As taxonomias são independentes e estão em diretórios separados.")


if __name__ == "__main__":
    main()



EXPERIMENTO: experiment_1_original_order
Ordem: ordem original do arquivo JSONL
Modelo: gpt-6.1-sol
Títulos: 1800
Tamanho dos lotes: 100
Total de lotes: 18
Tópicos existentes nesta taxonomia: 0
Lotes já concluídos: 0
Diretório: topic_generation\experiment_1_original_order


experiment_1_original_order:   0%|          | 0/18 [00:00<?, ?lote/s]


------------------------------------------------------------
LOTE 1/18
Títulos: 100
Tópicos antes da chamada: 0
------------------------------------------------------------
Enviando chamada à API...
Relatório de custos atualizado: topic_generation\experiment_1_original_order\costs.csv
Custo acumulado estimado (inclui tentativas falhas): US$ 0.01869350
Novos tópicos: 11
Total acumulado: 11
Tokens de entrada: 3,422
Tokens em cache: 0
Tokens de saída: 1,014
Custo estimado do lote: US$ 0.01869350

------------------------------------------------------------
LOTE 2/18
Títulos: 100
Tópicos antes da chamada: 11
------------------------------------------------------------
Enviando chamada à API...
Relatório de custos atualizado: topic_generation\experiment_1_original_order\costs.csv
Custo acumulado estimado (inclui tentativas falhas): US$ 0.03065700
Novos tópicos: 2
Total acumulado: 13
Tokens de entrada: 3,870
Tokens em cache: 0
Tokens de saída: 229
Custo estimado do lote: US$ 0.01196350

---

experiment_2_shuffled_order:   0%|          | 0/18 [00:00<?, ?lote/s]


------------------------------------------------------------
LOTE 1/18
Títulos: 100
Tópicos antes da chamada: 0
------------------------------------------------------------
Enviando chamada à API...
Relatório de custos atualizado: topic_generation\experiment_2_shuffled_order\costs.csv
Custo acumulado estimado (inclui tentativas falhas): US$ 0.02644350
Novos tópicos: 19
Total acumulado: 19
Tokens de entrada: 3,490
Tokens em cache: 0
Tokens de saída: 1,772
Custo estimado do lote: US$ 0.02644350

------------------------------------------------------------
LOTE 2/18
Títulos: 100
Tópicos antes da chamada: 19
------------------------------------------------------------
Enviando chamada à API...
Relatório de custos atualizado: topic_generation\experiment_2_shuffled_order\costs.csv
Custo acumulado estimado (inclui tentativas falhas): US$ 0.03747950
Novos tópicos: 1
Total acumulado: 20
Tokens de entrada: 3,939
Tokens em cache: 0
Tokens de saída: 119
Custo estimado do lote: US$ 0.01103600

---